In [ ]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.chart import BarChart, Reference
import string
from openpyxl.styles import Font
from os.path import split

def auto_excel(nombre_archivo):
    """Imput  sales_mes.xlsx / Output report_mes.xlsx"""
    archivo_excel = pd.read_excel("/content/supermarket_sales.xlsx")

    # Asegurarse de que la columna de fecha está en formato datetime
    archivo_excel['Date'] = pd.to_datetime(archivo_excel['Date'], format='%m/%d/%Y')

    # Obtener el mes de la fecha de la columna
    archivo_excel['Month'] = archivo_excel['Date'].dt.strftime('%B')  # Convierte el mes a texto en inglés

    # Extraer el nombre del mes del nombre del archivo
    mes_extension = nombre_archivo.split("_")[1].split(".")[0]  # ejemplo: 'enero'

    # Traducción de meses si es necesario (del español al inglés)
    meses_traduccion = {
        'enero': 'January', 'febrero': 'February', 'marzo': 'March',
        'abril': 'April', 'mayo': 'May', 'junio': 'June',
        'julio': 'July', 'agosto': 'August', 'septiembre': 'September',
        'octubre': 'October', 'noviembre': 'November', 'diciembre': 'December'
    }

    mes_en_ingles = meses_traduccion.get(mes_extension.lower(), '')

    # Filtrar los datos por el mes correspondiente
    archivo_excel = archivo_excel[archivo_excel['Month'] == mes_en_ingles]

    # Crear tabla pivote
    tabla_privote = pd.pivot_table(
        index="Gender", columns="Product line", values="Total", aggfunc="sum", data=archivo_excel).round(0)

    tabla_privote.to_excel(f"sales_{mes_extension}.xlsx", startrow=4, sheet_name="Report")

    # Cargar el archivo de Excel para aplicar el formato y gráficos
    wb = load_workbook(f"sales_{mes_extension}.xlsx")
    pestaña = wb["Report"]

    min_col = wb.active.min_column
    max_col = wb.active.max_column
    min_fila = wb.active.min_row
    max_fila = wb.active.max_row

    # Grafico
    barchart = BarChart()

    data = Reference(pestaña, min_col=min_col + 1, max_col=max_col, min_row=min_fila, max_row=max_fila)
    categorias = Reference(pestaña, min_col=min_col, max_col=min_col, min_row=min_fila + 1, max_row=max_fila)

    barchart.add_data(data, titles_from_data=True)
    barchart.set_categories(categorias)

    pestaña.add_chart(barchart, "B12")
    barchart.title = "Sales by Product line"
    barchart.style = 2

    abc = list(string.ascii_uppercase)
    abc_excel = abc[0:max_col]

    for i in abc_excel:
        if i != "A":
            pestaña[f"{i}{max_fila+1}"] = f"=SUM({i}{min_fila+1}:{i}{max_fila})"
            pestaña[f"{i}{max_fila+1}"].style = "Currency"

    pestaña[f"{abc_excel[0]}{max_fila+1}"] = "Total"

    pestaña["A1"] = "Reporte de ventas"
    pestaña["A2"] = mes_extension.capitalize()
    pestaña["A1"].font = Font("Arial", bold=True, size=20)
    pestaña["A2"].font = Font("Arial", bold=True, size=10)

    wb.save(f"report_{mes_extension}.xlsx")
    return

# Generar reportes mensuales
auto_excel("sales_enero.xlsx")
auto_excel("sales_febrero.xlsx")
